# Add BioSamples metadata

Using `mgnipy` we can also retrieve and integrate additional sample metadata from the [BioSamples](https://www.ebi.ac.uk/biosamples/) respository with the MGnify samples.
<br><br>

`mgnipy.collect.BioSampler()`

BioSampler can take a list of Run or Sample ENA accessions and retrieve rich metadata from BioSamples for a list of Run or Sample ENA/BioSamples accessions.

For this example we will get samples from MGnify using a `MGnifier`, passing the accessions to a `BioSampler` to get the BioSamples metadata.

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder. 
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [1]:
# uncomment below if colab
#!pip install mgnipy

## First getting MGnify samples

As done in the previous demo ["Finding sample details"](2_samples.ipynb), we'll start with initiating our MGnipy client with a cache, then building and executing our queries to the MGnify API.

In [2]:
from mgnipy import MGnipy

# init with cache
MG = MGnipy(cache_dir="temp_example")

# point to samples endpoint/get samples mgnifier
samples = MG.samples(search='cancer', page_size=5) # with search & pg sz params

# execute asynchronously 
async with MG:
    # get one page list of samples
    await samples.aget()
    # and if we want to get details for each sample in the list 
    await samples.aenrich_details()

Enriching sample details: 100%|██████████| 5/5 [00:00<00:00, 444.73it/s]


we now have our list of sample accessions to pass to our BioSampler

In [3]:
# check it out
print(samples.metadata.ids)

['SAMN03217341', 'SAMEA2247976', 'SAMEA2247987', 'SAMN03217340', 'SAMEA2247961']


## Now the BioSampler

we can access from our MGnipy client instance, which will also pass on our configuration such as the cache directory

In [4]:
# init biosampler from mgnipy client
bios = MG.biosampler(samples.metadata.ids)
# check it out
print(bios)

BioSampler with 5 sample_ids. 
Progress: 0 ids. 
Cache directory: temp_example/d6045f20348a1a934a32bb3442760a023a7f0723b16c7a7c301ae5183d8c8819 



Now let's execute the requests to the BioSamples API

In [5]:
# execute asynchronously and with context manager
async with bios:
    await bios.aenrich(limit=None)

Enriching biosamples: 100%|██████████| 5/5 [00:00<?, ?it/s]


## Merging metadata sets 
we can merge the metadata as we would any pandas or polars dataframe

In [6]:
samples.metadata.to_polars(expand_nested_dicts=['metadata']).join(
    bios.metadata.to_polars(), 
    left_on="accession", 
    right_on="SampleID",
    how="left"
)



accession,ena_accessions,sample_title,biome,updated_at,studies,metadata__ph,metadata__age,metadata__lat,metadata__lon,metadata__sex,metadata__tag,metadata__host,metadata__depth,metadata__status,metadata__strain,metadata__tax_id,metadata__country,metadata__datahub,metadata__disease,metadata__ecotype,metadata__isolate,metadata__serovar,metadata__variety,metadata__altitude,metadata__cultivar,metadata__germline,metadata__host_sex,metadata__keywords,metadata__location,metadata__salinity,metadata__serotype,metadata__cell_line,metadata__cell_type,metadata__checklist,metadata__dev_stage,metadata__elevation,…,metadata__broad_scale_environmental_context,GivenID,RunID,SRA accession,name,taxid,ENA-CHECKLIST,ENA-FIRST-PUBLIC,ENA-LAST-UPDATE,External Id,INSDC center name,INSDC first public,INSDC last update,INSDC status,Submitter Id,collection date,environment (biome),environment (feature),environment (material),"geographic location (countryand/orsea,region)",geographic location (latitude),geographic location (longitude),humam oral environmental package,investigation type,organism,project name,scientific_name,sequencing method,title,INSDC secondary accession,NCBI submission model,NCBI submission package,description,geo loc name,host,isolation source,lat lon
str,list[str],str,str,str,list[struct[6]],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SAMN03217341""","[""SAMN03217341"", ""SRS750277""]","""Oral cancer bacterial DNA""",null,"""2026-10-30T14:58:48.605000+00:…","[{""MGYS00002227"",[""SRP050159"", ""PRJNA268255""],""Oral cancer bacteriome - preliminary study Metagenome"",{""Oral"",""root:Host-associated:Human:Digestive system:Oral""},""2026-10-30T14:58:49.339000+00:00"",{""human oral metagenome"",""Jazan University"",""Oral cancer bacteriome - preliminary study Metagenome"",""PRJNA268255"",""Bacterial species present within oral cancer tissue"",""SRP050159""}}]","""""","""""","""30.0167""","""31.2167""","""""","""env_geo;env_geo:terrestrial""","""Homo sapiens""","""""","""public""","""""","""447426""","""Saudi Arabia: Jazan""","""dcc_metagenome""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""30.0167 N 31.2167 E""","""""","""""","""""","""""","""""","""""","""""",…,"""""","""SAMN03217341""",null,"""SRS750277""","""CaPre02""",447426,null,null,null,"""SAMN03217341""","""Jazan University""","""2014-11-23T00:00:00Z""","""2019-06-20T17:09:25.325Z""","""live""",null,"""2011""",null,null,null,null,null,null,null,null,"""human oral metagenome""",null,null,null,"""Oral cancer bacterial DNA""","""SRS750277""","""Metagenome or environmental""","""Metagenome.environmental.1.0""","""Oral cancer bacterial DNA""","""Saudi Arabia: Jazan""","""Homo sapiens""","""Oral cancer tissue""","""30.0167 N 31.2167 E"""
"""SAMEA2247976""","[""ERS370386"", ""SAMEA2247976""]","""Oral Cancer Microbiome""",null,"""2026-10-30T14:58:06.003000+00:…","[{""MGYS00002210"",[""ERP004294"", ""PRJEB4953""],""This study develops a framework for exploiting the oral microbiome for monitoring oral cancer development, progression and recurrence."",{""Oral"",""root:Host-associated:Human:Digestive system:Oral""},""2026-10-30T14:58:05.984000+00:00"",{""Oral Microbiome for cancer development, progression and recurrence"",""Second Genome Inc.;SECONDGENOME"",""This study develops a framework for exploiting the oral microbiome for monitoring oral cancer development, progression and recurrence."",""PRJEB4953"",""Individual bacteria and shifts in the composition of the microbiome have been associated with human diseases including cancer. To investigate changes in the microbiome associated with oral cancers, we profiled cancers and anatomically matched contralateral normal tissue from the same patient by sequencing 16S rDNA hypervariable r

if wanting to cleanup the "temp_example" cache

In [7]:
#MG.clear_subcaches()